# Semanas 10–12 — Modelos y optimización
**Familias:** lineales, árboles/ensambles, boosting. Comparación temporal sobre marzo; **no se usa julio/test final**. MAE es la métrica de selección.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src.data import download_yellow_taxi_months, load_month
from src.modeling import prepare, sample_month, FEATURES, models, metrics
from sklearn.model_selection import ParameterGrid
import json
SEED=42
MAX_ROWS_PER_MONTH=100000  # Cambiar a None para usar todos los registros (requiere memoria)
MONTHS=["2026-01","2026-02","2026-03"]
# Descarga automática; requiere conexión y espacio en disco
paths=download_yellow_taxi_months(MONTHS, ROOT/'data')
raw={m:load_month(p,columns=["tpep_pickup_datetime","tpep_dropoff_datetime","passenger_count","PULocationID","DOLocationID"]) for m,p in paths.items()}
frames={m:sample_month(prepare(df),MAX_ROWS_PER_MONTH,SEED) for m,df in raw.items()}
train=pd.concat([frames["2026-01"],frames["2026-02"]],ignore_index=True)
val=frames["2026-03"]
X_train,y_train=train[FEATURES],train["duration_min"]
X_val,y_val=val[FEATURES],val["duration_min"]
print('Train:',len(train),'Validación:',len(val))

In [ ]:
from src.modeling import models,metrics
import time
rows=[]
for name,model in models(SEED).items():
    start=time.time()
    model.fit(X_train,y_train)
    result={"modelo":name,**metrics(y_val,model.predict(X_val)),"segundos":round(time.time()-start,1)}
    rows.append(result)
    print(result)
comparison=pd.DataFrame(rows).sort_values('MAE_min')
display(comparison)
(ROOT/'outputs').mkdir(exist_ok=True)
comparison.to_csv(ROOT/'outputs'/'comparacion_modelos_semana11.csv',index=False)

## Semana 12 — Búsqueda de hiperparámetros
Búsqueda acotada para tres familias. Se usa enero para ajustar y febrero para seleccionar hiperparámetros; marzo se reserva para comparar configuraciones ya elegidas. No se utiliza el test final.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from src.modeling import preprocessing
X_jan,y_jan=frames['2026-01'][FEATURES],frames['2026-01']['duration_min']
X_feb,y_feb=frames['2026-02'][FEATURES],frames['2026-02']['duration_min']
search_spaces={
 'ridge': (Ridge, {'alpha':[1.0,10.0,100.0]}),
 'arbol': (DecisionTreeRegressor, {'max_depth':[8,14], 'min_samples_leaf':[20,80]}),
 'gradient_boosting': (HistGradientBoostingRegressor, {'max_iter':[80,150], 'max_leaf_nodes':[15,31]})
}
selected=[]
for family,(cls,grid) in search_spaces.items():
    best=None
    for params in ParameterGrid(grid):
        opts=dict(params)
        if family!='ridge':opts['random_state']=SEED
        estimator=Pipeline([('prep',preprocessing()),('model',cls(**opts))])
        estimator.fit(X_jan,y_jan)
        score=metrics(y_feb,estimator.predict(X_feb))['MAE_min']
        if best is None or score<best['MAE_feb']:
            best={'familia':family,'parametros':params,'MAE_feb':score}
    selected.append(best)
    print(best)
(ROOT/'outputs'/'hiperparametros_semana12.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')

In [ ]:
# Reentrenamiento enero-febrero y evaluación marzo, una vez fijados los hiperparámetros
final_rows=[]
for choice in selected:
    cls=search_spaces[choice['familia']][0]
    kwargs=dict(choice['parametros'])
    if choice['familia']!='ridge':kwargs['random_state']=SEED
    model=Pipeline([('prep',preprocessing()),('model',cls(**kwargs))])
    model.fit(X_train,y_train)
    final_rows.append({'familia':choice['familia'],**metrics(y_val,model.predict(X_val))})
final_comparison=pd.DataFrame(final_rows).sort_values('MAE_min')
display(final_comparison)
final_comparison.to_csv(ROOT/'outputs'/'modelos_optimizados_semana12.csv',index=False)
print('Mejor familia en validación:',final_comparison.iloc[0]['familia'])

**Interpretación:** seleccionar el menor MAE de validación. RMSE indica sensibilidad a errores grandes y R² la mejora respecto a la predicción constante del promedio de validación. No afirmar resultados numéricos sin ejecutar el notebook sobre los Parquet oficiales.